# Combined Master Recode Notebook

This notebook only reclassifies the final `orientation` and `secondary_category` columns from the already-exported combined master workbook.

It does **not** rerun PDF extraction, chunking, retrieval, or the first-pass relevance classification. The idea is to save tokens and only tighten the two fields that received feedback:

- `orientation` must end up as exactly `proactive`, `reactive`, or `descriptive`
- `secondary_category` must end up as either `0` or one of the exact E1-E7 labels

The notebook uses the final Excel file as input, calls the model with a worker pool, stores a resumable JSONL cache, and writes a new output workbook with the recoded values.

In [ ]:
from __future__ import annotations

import json
import logging
import os
import pickle
import re
import textwrap
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from hashlib import sha1
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import pandas as pd
from IPython.display import display
from openai import OpenAI
from pandas.api.types import is_string_dtype
from tqdm.auto import tqdm
from dotenv import load_dotenv

load_dotenv(override=True)
load_dotenv(Path("/Users/paulkoslowsky/Github/Jakop") / ".env")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)
logger = logging.getLogger("semiconductor_recode")

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", 160)


In [ ]:
CONFIG: Dict[str, Any] = {
    "input_excel_path": Path("outputs/final_pipeline/combined/combined_master.xlsx"),
    "output_dir": Path("outputs/final_pipeline/combined/recode_orientation_secondary"),
    "output_excel_path": Path("outputs/final_pipeline/combined/recode_orientation_secondary/combined_master_recoded.xlsx"),
    "output_pickle_path": Path("outputs/final_pipeline/combined/recode_orientation_secondary/combined_master_recoded.pkl"),
    "changed_rows_excel_path": Path("outputs/final_pipeline/combined/recode_orientation_secondary/combined_master_changed_rows.xlsx"),
    "cache_path": Path("outputs/final_pipeline/combined/recode_orientation_secondary/recode_cache.jsonl"),
    "openai_model_name": "gpt-5.4-mini",
    "openai_api_key_env_var": "OPENAI_API_KEY",
    "max_workers": 20,
    "max_attempts": 2,
    "max_rows": None,
    "selected_companies": None,
}

CONFIG["output_dir"].mkdir(parents=True, exist_ok=True)

print("Active recode configuration:")
for key, value in CONFIG.items():
    print(f"- {key}: {value}")


In [ ]:
CONTROLLED_STRATEGY_CATEGORIES: List[str] = [
    "E1 Operational Buffers",
    "E2 Footprint Diversification",
    "E3 Supply Option Diversification",
    "E4 Robust Distribution",
    "E5 Product Standardisation",
    "E6 Partner Network Strengthening",
    "E7 SCRM and Visibility",
]

ALLOWED_ORIENTATION_VALUES: List[str] = [
    "proactive",
    "reactive",
    "descriptive",
]

COMPANY_SCOPE_RULES: Dict[str, str] = {
    "Samsung": (
        "For Samsung, only interpret the passage in the DS Division, semiconductor, memory, or foundry context. "
        "Ignore other business units unless the statement clearly refers to the full company and is materially relevant to semiconductor supply-chain strategy."
    ),
    "MediaTek": (
        "For MediaTek, only interpret the passage in MediaTek Inc.'s own semiconductor business context. "
        "Ignore unrelated subsidiary content."
    ),
}

EXCEL_ILLEGAL_CHARACTER_RE = re.compile(r"[\x00-\x08\x0B-\x0C\x0E-\x1F]")


def normalise_whitespace(text: str) -> str:
    """Collapse messy whitespace while keeping line breaks readable."""

    text = text.replace("\x00", " ")
    text = re.sub(r"\r\n?", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def normalize_for_matching(text: str) -> str:
    """Normalize strings for rule-based comparisons."""

    text = normalise_whitespace(text)
    text = re.sub(r"\s+", " ", text)
    return text.lower().strip()


def normalize_to_controlled_category(raw_value: Optional[str]) -> Optional[str]:
    """Map loose model wording back into the exact E1-E7 label set."""

    if raw_value is None:
        return None

    lowered = normalize_for_matching(str(raw_value)).replace("_", " ")
    exact_map = {
        normalize_for_matching(category).replace("_", " "): category
        for category in CONTROLLED_STRATEGY_CATEGORIES
    }
    if lowered in exact_map:
        return exact_map[lowered]

    pattern_map: List[Tuple[List[str], str]] = [
        (["operational buffer", "inventory", "safety stock", "buffer stock", "capacity buffer"], "E1 Operational Buffers"),
        (["footprint", "geographic diversification", "manufacturing expansion", "fab location", "nearshoring", "regional diversification"], "E2 Footprint Diversification"),
        (["dual sourcing", "alternative supplier", "second source", "backup supply", "supplier diversification", "sourcing"], "E3 Supply Option Diversification"),
        (["distribution", "logistics", "freight", "shipment", "channel management", "transport"], "E4 Robust Distribution"),
        (["standardisation", "standardization", "sku", "common design", "platform architecture", "modular design"], "E5 Product Standardisation"),
        (["partnership", "collaboration", "long term supply", "capacity reservation", "ecosystem", "foundry collaboration"], "E6 Partner Network Strengthening"),
        (["risk management", "resilience", "visibility", "business continuity", "geopolitical", "trade restrictions", "monitoring", "risk assessment"], "E7 SCRM and Visibility"),
    ]

    for keywords, normalized_category in pattern_map:
        if any(keyword in lowered for keyword in keywords):
            return normalized_category

    return None


def normalize_secondary_category(raw_value: Optional[str]) -> str:
    """Normalize `secondary_category` to either 0 or the exact E1-E7 set."""

    if raw_value is None:
        return "0"

    lowered = normalize_for_matching(str(raw_value)).replace("_", " ")
    if lowered in {"0", "none", "null", "na", "n a", "missing", "other", "n/a", "no secondary category"}:
        return "0"

    normalized_category = normalize_to_controlled_category(raw_value)
    return normalized_category if normalized_category is not None else "0"


def normalize_orientation(raw_value: Optional[str]) -> str:
    """Normalize `orientation` to proactive, reactive, or descriptive."""

    if raw_value is None:
        return "descriptive"

    lowered = normalize_for_matching(str(raw_value)).replace("_", " ")

    proactive_keywords = [
        "proactive", "anticipatory", "preventive", "preparedness", "preemptive", "forward looking", "long term", "capacity build", "invest in resilience", "build capability"
    ]
    reactive_keywords = [
        "reactive", "response", "responding", "mitigation after", "contingency response", "crisis response", "recovery", "after disruption", "shortage response"
    ]
    descriptive_keywords = [
        "descriptive", "observational", "context", "background", "narrative", "general", "informational", "disclosure", "risk factor", "market overview"
    ]

    if any(keyword in lowered for keyword in proactive_keywords):
        return "proactive"
    if any(keyword in lowered for keyword in reactive_keywords):
        return "reactive"
    if any(keyword in lowered for keyword in descriptive_keywords):
        return "descriptive"

    return "descriptive"


def _sanitize_for_excel(value: Any) -> Any:
    """Remove characters that Excel/openpyxl cannot write safely."""

    if not isinstance(value, str):
        return value

    cleaned_value = EXCEL_ILLEGAL_CHARACTER_RE.sub("", value)
    return cleaned_value[:32767]


def sanitize_dataframe_for_excel(dataframe: pd.DataFrame) -> pd.DataFrame:
    """Return a copy with all string-like columns cleaned for openpyxl."""

    sanitized_df = dataframe.copy()
    for column_name in sanitized_df.columns:
        if is_string_dtype(sanitized_df[column_name]):
            sanitized_df[column_name] = sanitized_df[column_name].map(_sanitize_for_excel)
    return sanitized_df


def build_row_key(row: pd.Series) -> str:
    """Create a stable row key so cached recodes can be reused across reruns."""

    key_parts = [
        str(row.get("firm_name", "")),
        str(row.get("fiscal_year", "")),
        str(row.get("chunk_id", "")),
        str(row.get("retrieval_query", "")),
        str(row.get("retrieval_rank", "")),
    ]
    return sha1("||".join(key_parts).encode("utf-8")).hexdigest()


In [ ]:
EXPECTED_RECODE_KEYS: List[str] = [
    "secondary_category",
    "orientation",
    "confidence",
]


def get_openai_client(api_key_env_var: str) -> OpenAI:
    """Create an OpenAI client from an environment variable."""

    api_key = os.getenv(api_key_env_var)
    if not api_key:
        raise EnvironmentError(
            f"Environment variable '{api_key_env_var}' is not set. "
            "Set your API key before running recoding."
        )
    return OpenAI(api_key=api_key)


_thread_local_state = threading.local()
_cache_write_lock = threading.Lock()


def get_thread_local_openai_client(api_key_env_var: str) -> OpenAI:
    """Return one OpenAI client per worker thread for safer concurrent requests."""

    client = getattr(_thread_local_state, "openai_client", None)
    client_env = getattr(_thread_local_state, "api_key_env_var", None)

    if client is None or client_env != api_key_env_var:
        client = get_openai_client(api_key_env_var)
        _thread_local_state.openai_client = client
        _thread_local_state.api_key_env_var = api_key_env_var

    return client


def extract_json_object(text: str) -> Optional[Dict[str, Any]]:
    """Extract the first JSON object from a text response."""

    text = text.strip()
    if not text:
        return None

    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if not match:
        return None

    try:
        return json.loads(match.group(0))
    except json.JSONDecodeError:
        return None


def validate_recode_payload(payload: Dict[str, Any]) -> Dict[str, Any]:
    """Validate the raw recode payload before normalization."""

    if set(payload.keys()) != set(EXPECTED_RECODE_KEYS):
        raise ValueError(
            f"Invalid JSON keys. Expected exactly {EXPECTED_RECODE_KEYS}, got {list(payload.keys())}"
        )

    if payload["orientation"] is not None and not isinstance(payload["orientation"], str):
        raise ValueError("'orientation' must be a string or null.")

    if payload["secondary_category"] is not None and not isinstance(payload["secondary_category"], str):
        raise ValueError("'secondary_category' must be a string or null.")

    if not isinstance(payload["confidence"], (int, float)):
        raise ValueError("'confidence' must be numeric.")

    confidence_value = float(payload["confidence"])
    if not 0 <= confidence_value <= 1:
        raise ValueError("'confidence' must be between 0 and 1.")

    return {
        "secondary_category": normalize_secondary_category(payload["secondary_category"]),
        "orientation": normalize_orientation(payload["orientation"]),
        "confidence": confidence_value,
    }


def build_recode_prompt(row: pd.Series) -> str:
    """Build the recoding prompt for one final export row."""

    allowed_categories_text = "\n".join(f"- {category}" for category in CONTROLLED_STRATEGY_CATEGORIES)
    company_scope_text = COMPANY_SCOPE_RULES.get(str(row.get("firm_name", "")), "")

    instructions = textwrap.dedent(
        f"""
        You are post-processing an already relevant annual-report passage for a semiconductor supply-chain thesis.

        Your task is to recode only two fields while keeping the already-assigned primary category fixed.

        Return exactly one JSON object and nothing else.
        The JSON object must contain exactly these keys:
        secondary_category
        orientation
        confidence

        Rules:
        1. Do not change or reinterpret the fixed primary category beyond using it as context.
        2. `secondary_category` must be either:
           - `0` if there is no clearly distinct second strategy visible in the passage
           - or one of these exact labels:
        {allowed_categories_text}
        3. Use `0` unless a second E-category is clearly present in addition to the primary one.
        4. `orientation` must be exactly one of:
           - proactive
           - reactive
           - descriptive
        5. Interpret orientation as follows:
           - proactive = anticipatory capability-building, prevention, preparedness, diversification, monitoring, buffers, or forward-looking resilience action before disruption materializes
           - reactive = response, recovery, remediation, or mitigation after a disruption, shortage, sanction, accident, or other problem has already materialized
           - descriptive = context, disclosure, background, market overview, risk-factor wording, or general strategy narration without a clear proactive or reactive action posture
        6. Prefer `descriptive` for generic risk disclosures, broad outlook text, and explanatory background passages.
        7. Confidence must be numeric between 0 and 1.
        """
    ).strip()

    metadata_block = textwrap.dedent(
        f"""
        Metadata:
        - firm_name: {row.get("firm_name")}
        - fiscal_year: {row.get("fiscal_year")}
        - source_file: {row.get("source_file")}
        - page_number: {row.get("page_number")}
        - section: {row.get("section")}
        - fixed_primary_strategy_category: {row.get("strategy_category")}
        - current_main_point: {row.get("main_point")}
        - current_geopolitical_trigger: {row.get("geopolitical_trigger")}
        """
    ).strip()

    scope_block = f"Company-specific scope rule:\n{company_scope_text}" if company_scope_text else ""

    passage_text = normalise_whitespace(str(row.get("passage", "")))
    return f"{instructions}\n\n{metadata_block}\n\n{scope_block}\n\nPassage:\n{passage_text}"


def request_openai_json_response(client: OpenAI, model_name: str, prompt: str) -> str:
    """Request a raw model response from OpenAI."""

    response = client.responses.create(
        model=model_name,
        input=prompt,
    )
    return response.output_text


def recode_row_with_retries(
    row: pd.Series,
    client: OpenAI,
    model_name: str,
    max_attempts: int = 2,
) -> Tuple[Optional[Dict[str, Any]], Optional[str]]:
    """Recode one row and retry once with a repair prompt if needed."""

    prompt = build_recode_prompt(row)
    raw_response_text = ""

    for attempt_number in range(1, max_attempts + 1):
        try:
            if attempt_number == 1:
                raw_response_text = request_openai_json_response(client, model_name, prompt)
            else:
                repair_prompt = textwrap.dedent(
                    f"""
                    The following output was invalid. Repair it into valid JSON only.

                    Required keys:
                    {EXPECTED_RECODE_KEYS}

                    Rules:
                    - return exactly one JSON object
                    - do not add markdown or commentary
                    - preserve the original recoding intent if possible
                    - orientation must be proactive, reactive, or descriptive
                    - secondary_category must be 0 or one of the exact E1-E7 labels
                    - confidence must be numeric between 0 and 1

                    Invalid output:
                    {raw_response_text}
                    """
                ).strip()
                raw_response_text = request_openai_json_response(client, model_name, repair_prompt)

            parsed_payload = extract_json_object(raw_response_text)
            if parsed_payload is None:
                raise ValueError("No valid JSON object found in the model response.")

            return validate_recode_payload(parsed_payload), None

        except Exception as error:  # noqa: BLE001
            if attempt_number == max_attempts:
                return None, f"Recoding failed after {max_attempts} attempts: {error}"

    return None, "Unexpected recoding failure."


def recode_row_record(
    row_dict: Dict[str, Any],
    model_name: str,
    api_key_env_var: str,
    max_attempts: int,
) -> Dict[str, Any]:
    """Process one row and return a cache-ready result record."""

    row_series = pd.Series(row_dict)
    row_key = row_series["recode_row_key"]
    client = get_thread_local_openai_client(api_key_env_var)

    payload, error_message = recode_row_with_retries(
        row=row_series,
        client=client,
        model_name=model_name,
        max_attempts=max_attempts,
    )

    if payload is None:
        return {
            "recode_row_key": row_key,
            "recode_failed": True,
            "recode_error": error_message,
            "secondary_category": None,
            "orientation": None,
            "recode_confidence": None,
        }

    return {
        "recode_row_key": row_key,
        "recode_failed": False,
        "recode_error": None,
        "secondary_category": payload["secondary_category"],
        "orientation": payload["orientation"],
        "recode_confidence": payload["confidence"],
    }


def load_cache(cache_path: Path) -> Dict[str, Dict[str, Any]]:
    """Load the resumable JSONL cache if it exists."""

    if not cache_path.exists():
        return {}

    cache: Dict[str, Dict[str, Any]] = {}
    with cache_path.open("r", encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if not line:
                continue
            payload = json.loads(line)
            cache[payload["recode_row_key"]] = payload
    return cache


def append_cache_record(cache_path: Path, payload: Dict[str, Any]) -> None:
    """Append one result record to the resumable JSONL cache."""

    with _cache_write_lock:
        with cache_path.open("a", encoding="utf-8") as handle:
            handle.write(json.dumps(payload, ensure_ascii=False) + "\n")


In [ ]:
combined_df = pd.read_excel(CONFIG["input_excel_path"])

required_columns = [
    "firm_name",
    "fiscal_year",
    "section",
    "page_number",
    "passage",
    "strategy_category",
    "secondary_category",
    "orientation",
    "main_point",
    "geopolitical_trigger",
    "source_file",
    "chunk_id",
    "retrieval_query",
    "retrieval_rank",
]

missing_columns = [column for column in required_columns if column not in combined_df.columns]
if missing_columns:
    raise ValueError(f"Missing required columns in combined master Excel: {missing_columns}")

working_df = combined_df.copy()

if CONFIG["selected_companies"]:
    working_df = working_df.loc[
        working_df["firm_name"].isin(CONFIG["selected_companies"])
    ].copy()

if CONFIG["max_rows"] is not None:
    working_df = working_df.head(int(CONFIG["max_rows"])).copy()

working_df["recode_row_key"] = working_df.apply(build_row_key, axis=1)
working_df["orientation_original"] = working_df["orientation"]
working_df["secondary_category_original"] = working_df["secondary_category"]

print(f"Loaded {len(working_df)} rows for recoding from: {CONFIG['input_excel_path'].resolve()}")
display(working_df.head(3))


In [ ]:
cached_results = load_cache(CONFIG["cache_path"])
print(f"Existing cached recodes found: {len(cached_results)}")

rows_to_process_df = working_df.loc[
    ~working_df["recode_row_key"].isin(cached_results.keys())
].copy()

print(f"Rows still needing API calls: {len(rows_to_process_df)}")

if not rows_to_process_df.empty:
    row_dicts = rows_to_process_df.to_dict(orient="records")

    with ThreadPoolExecutor(max_workers=CONFIG["max_workers"]) as executor:
        future_to_key = {
            executor.submit(
                recode_row_record,
                row_dict=row_dict,
                model_name=CONFIG["openai_model_name"],
                api_key_env_var=CONFIG["openai_api_key_env_var"],
                max_attempts=CONFIG["max_attempts"],
            ): row_dict["recode_row_key"]
            for row_dict in row_dicts
        }

        for future in tqdm(
            as_completed(future_to_key),
            total=len(future_to_key),
            desc=f"Recoding rows ({CONFIG['max_workers']} workers)",
        ):
            row_key = future_to_key[future]
            result_payload = future.result()
            cached_results[row_key] = result_payload
            append_cache_record(CONFIG["cache_path"], result_payload)

result_df = pd.DataFrame(cached_results.values())
if result_df.empty:
    raise ValueError("No recode results were available after processing.")

working_df = working_df.merge(result_df, on="recode_row_key", how="left", suffixes=("", "_recode"))

successful_mask = working_df["recode_failed"] == False  # noqa: E712
working_df.loc[successful_mask, "orientation"] = working_df.loc[successful_mask, "orientation_recode"]
working_df.loc[successful_mask, "secondary_category"] = working_df.loc[successful_mask, "secondary_category_recode"]

working_df = working_df.drop(columns=["orientation_recode", "secondary_category_recode"])

print(f"Successful recodes: {int(successful_mask.fillna(False).sum())}")
print(f"Failed recodes kept at original values: {int(working_df['recode_failed'].fillna(True).sum())}")


In [ ]:
changed_rows_df = working_df.loc[
    (working_df["orientation"] != working_df["orientation_original"])
    | (working_df["secondary_category"] != working_df["secondary_category_original"])
].copy()

sanitized_output_df = sanitize_dataframe_for_excel(working_df)
sanitized_changed_rows_df = sanitize_dataframe_for_excel(changed_rows_df)

sanitized_output_df.to_excel(CONFIG["output_excel_path"], index=False, engine="openpyxl")
sanitized_changed_rows_df.to_excel(CONFIG["changed_rows_excel_path"], index=False, engine="openpyxl")

with CONFIG["output_pickle_path"].open("wb") as handle:
    pickle.dump(working_df, handle)

print(f"Recoded workbook written to: {CONFIG['output_excel_path'].resolve()}")
print(f"Changed rows workbook written to: {CONFIG['changed_rows_excel_path'].resolve()}")
print(f"Recoded pickle written to: {CONFIG['output_pickle_path'].resolve()}")
print(f"Resumable cache path: {CONFIG['cache_path'].resolve()}")

print("\nOrientation counts")
display(working_df["orientation"].fillna("Missing").value_counts(dropna=False).rename("count").to_frame())

print("\nSecondary category counts")
display(working_df["secondary_category"].fillna("Missing").value_counts(dropna=False).rename("count").to_frame())

print("\nRows changed")
display(changed_rows_df[[
    "firm_name",
    "fiscal_year",
    "strategy_category",
    "secondary_category_original",
    "secondary_category",
    "orientation_original",
    "orientation",
    "recode_confidence",
    "recode_failed",
]].head(20))
